# LLM Microstructure Forecasting - 3-Arm Pipeline\n
\n
**Goal:** Test if providing market microstructure data improves LLM forecast accuracy\n
\n
**Design:**\n
- Load N markets from CSV\n
- Query each market 3 times with different contexts:\n
  - **Baseline**: Question only\n
  - **Volume**: Question + volume data\n
  - **Microstructure**: Question + volume + order book data\n
- Compare Brier score & log loss across arms

In [17]:
import os
import re
import asyncio
from datetime import datetime, UTC
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import AsyncOpenAI
from sklearn.metrics import brier_score_loss, log_loss

load_dotenv()

# Config
CSV_PATH = "data/events_2wk.csv"
OUT_PATH = "data/pilot_results.csv"
MODEL = "llama-3.1-8b-instant"
CONCURRENCY = 8
N_MARKETS = 20  # Change to 349 for full run
TEMPERATURE = 0

# Initialize Groq client
client = AsyncOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],
)

print(f"✓ Config loaded: {N_MARKETS} markets, concurrency={CONCURRENCY}")

✓ Config loaded: 20 markets, concurrency=8


In [18]:
# Load markets data
df_markets = pd.read_csv(CSV_PATH).head(N_MARKETS)
print(f"✓ Loaded {len(df_markets)} markets from {CSV_PATH}\n")
print("Columns:", list(df_markets.columns))
print("\nSample market:")
df_markets.head(3)

✓ Loaded 20 markets from data/events_2wk.csv

Columns: ['category', 'tags', 'series_ticker', 'event_ticker', 'event_title', 'close_time', 'time_to_close_hours', 'num_markets', 'total_volume', 'total_volume_24h', 'total_open_interest']

Sample market:


,category,tags,series_ticker,event_ticker,event_title,close_time,time_to_close_hours,num_markets,total_volume,total_volume_24h,total_open_interest
0,Climate and Weather,Daily temperature,KXHIGHLAX,KXHIGHLAX-26FEB05,"Highest temperature in LA on Feb 5, 2026?",2026-02-06 07:59:00+00:00,5.2,6,626806,594759,418680
1,Climate and Weather,Daily temperature,KXHIGHNY,KXHIGHNY-26FEB05,"Highest temperature in NYC on Feb 5, 2026?",2026-02-06 04:59:00+00:00,2.2,6,335417,317601,171132
2,Climate and Weather,Climate change,KXHMONTH,KXHMONTH-26JAN,This Jan 2026 is the hottest January ever?,2026-02-16 04:59:00+00:00,242.2,1,239162,7064,110156


In [19]:
# Prompt templates for each arm

SYSTEM_PROMPT = "You are a forecasting expert. Return only a probability between 0 and 1, nothing else."

def build_prompt_baseline(row):
    """Arm 1: Question only"""
    return f"""Forecast the probability this prediction market resolves YES.

Question: {row['event_title']}

Probability (0 to 1):"""

def build_prompt_volume(row):
    """Arm 2: Question + volume data"""
    return f"""Forecast the probability this prediction market resolves YES.

Question: {row['event_title']}

Market Data:
- Total Volume: ${row['total_volume']:,.0f}
- 24h Volume: ${row['total_volume_24h']:,.0f}

Probability (0 to 1):"""

def build_prompt_microstructure(row):
    """Arm 3: Question + volume + microstructure"""
    return f"""Forecast the probability this prediction market resolves YES.

Question: {row['event_title']}

Market Data:
- Total Volume: ${row['total_volume']:,.0f}
- 24h Volume: ${row['total_volume_24h']:,.0f}
- Open Interest: ${row['total_open_interest']:,.0f}
- Number of Sub-markets: {row['num_markets']}
- Time to Close: {row['time_to_close_hours']:.1f} hours

Probability (0 to 1):"""

# Arm definitions
ARMS = {
    'baseline': build_prompt_baseline,
    'volume': build_prompt_volume,
    'microstructure': build_prompt_microstructure,
}

print("✓ Defined 3 arms: baseline, volume, microstructure")
print("\nExample baseline prompt:")
print(build_prompt_baseline(df_markets.iloc[0]))

✓ Defined 3 arms: baseline, volume, microstructure

Example baseline prompt:
Forecast the probability this prediction market resolves YES.

Question: Highest temperature in LA on Feb 5, 2026?

Probability (0 to 1):


In [20]:
# API interaction functions

def parse_probability(text: str) -> float:
    """Extract probability from LLM response"""
    if not text:
        return 0.5
    # Match decimal (0.73) or percentage (73%)
    match = re.search(r'([01]?\.\d+|[01])', text.strip())
    if match:
        p = float(match.group(1))
        return p if p <= 1 else p / 100
    return 0.5  # Default if unparseable

async def query_market(row, arm_name, semaphore):
    """Query LLM for one market with one arm"""
    async with semaphore:
        prompt_fn = ARMS[arm_name]
        prompt = prompt_fn(row)
        
        try:
            response = await client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt}
                ],
                temperature=TEMPERATURE,
            )
            raw = response.choices[0].message.content.strip()
            p_yes = parse_probability(raw)
            error = None
        except Exception as e:
            raw = None
            p_yes = None
            error = str(e)
        
        return {
            'timestamp_utc': datetime.now(UTC).isoformat(),
            'model': MODEL,
            'arm': arm_name,
            'ticker': row['event_ticker'],
            'title': row['event_title'],
            'raw_response': raw,
            'p_yes': p_yes,
            'error': error
        }

print("✓ API functions ready")

✓ API functions ready


In [21]:
# Run pipeline: query all markets × all arms

async def run_pipeline(df_markets):
    semaphore = asyncio.Semaphore(CONCURRENCY)
    tasks = []
    
    # Create task for each market × arm combination
    for _, row in df_markets.iterrows():
        for arm_name in ARMS.keys():
            task = query_market(row, arm_name, semaphore)
            tasks.append(task)
    
    print(f"Starting {len(tasks)} API calls ({len(df_markets)} markets × 3 arms)...")
    results = await asyncio.gather(*tasks)
    return pd.DataFrame(results)

# Execute
df_results = await run_pipeline(df_markets)

# Save results
df_results.to_csv(OUT_PATH, index=False)

print(f"\n✓ Completed {len(df_results)} predictions")
print(f"✓ Saved to {OUT_PATH}\n")

print("Breakdown by arm:")
print(df_results['arm'].value_counts())
print(f"\nErrors: {df_results['error'].notna().sum()}")
print(f"\nSample results:")
df_results[['ticker', 'arm', 'p_yes', 'raw_response']].head(9)

Starting 60 API calls (20 markets × 3 arms)...

✓ Completed 60 predictions
✓ Saved to data/pilot_results.csv

Breakdown by arm:
arm
baseline          20
volume            20
microstructure    20
Name: count, dtype: int64

Errors: 20

Sample results:


,ticker,arm,p_yes,raw_response
0,KXHIGHLAX-26FEB05,baseline,0.620,0.62
1,KXHIGHLAX-26FEB05,volume,0.730,0.73
2,KXHIGHLAX-26FEB05,microstructure,0.623,0.623
3,KXHIGHNY-26FEB05,baseline,0.320,0.32
4,KXHIGHNY-26FEB05,volume,0.530,0.53
5,KXHIGHNY-26FEB05,microstructure,0.623,0.623
6,KXHMONTH-26JAN,baseline,0.230,0.23
7,KXHMONTH-26JAN,volume,0.320,0.32
8,KXHMONTH-26JAN,microstructure,0.320,0.32


In [22]:
# Add fake outcomes (random 0/1) for testing
# In production, replace this with actual market resolutions from API

np.random.seed(42)

# Each ticker gets one outcome (same across all arms)
unique_tickers = df_results['ticker'].unique()
outcome_map = {ticker: np.random.randint(0, 2) for ticker in unique_tickers}
df_results['actual_outcome'] = df_results['ticker'].map(outcome_map)

print(f"✓ Added fake outcomes for {len(unique_tickers)} markets")
print(f"  Resolution: {outcome_map[unique_tickers[0]]} (0=NO, 1=YES)\n")

# Save with outcomes
df_results.to_csv('data/pilot_results_with_outcomes.csv', index=False)
print(f"✓ Saved to data/pilot_results_with_outcomes.csv\n")

df_results[['ticker', 'arm', 'p_yes', 'actual_outcome']].head(9)

✓ Added fake outcomes for 20 markets
  Resolution: 0 (0=NO, 1=YES)

✓ Saved to data/pilot_results_with_outcomes.csv



,ticker,arm,p_yes,actual_outcome
0,KXHIGHLAX-26FEB05,baseline,0.620,0
1,KXHIGHLAX-26FEB05,volume,0.730,0
2,KXHIGHLAX-26FEB05,microstructure,0.623,0
3,KXHIGHNY-26FEB05,baseline,0.320,1
4,KXHIGHNY-26FEB05,volume,0.530,1
5,KXHIGHNY-26FEB05,microstructure,0.623,1
6,KXHMONTH-26JAN,baseline,0.230,0
7,KXHMONTH-26JAN,volume,0.320,0
8,KXHMONTH-26JAN,microstructure,0.320,0


In [23]:
# Evaluate each arm

print("=" * 80)
print("EVALUATION: Forecast Accuracy by Arm")
print("=" * 80)

results_summary = []

for arm_name in ['baseline', 'volume', 'microstructure']:
    df_arm = df_results[df_results['arm'] == arm_name].copy()
    df_arm = df_arm.dropna(subset=['p_yes', 'actual_outcome'])
    
    if len(df_arm) == 0:
        print(f"\n{arm_name.upper()}: No valid predictions")
        continue
    
    # Metrics
    brier = brier_score_loss(df_arm['actual_outcome'], df_arm['p_yes'])
    logloss = log_loss(df_arm['actual_outcome'], df_arm['p_yes'])
    
    df_arm['predicted'] = (df_arm['p_yes'] >= 0.5).astype(int)
    accuracy = (df_arm['predicted'] == df_arm['actual_outcome']).mean()
    correct = (df_arm['predicted'] == df_arm['actual_outcome']).sum()
    
    results_summary.append({
        'arm': arm_name,
        'brier_score': brier,
        'log_loss': logloss,
        'accuracy': accuracy,
        'n_markets': len(df_arm)
    })
    
    print(f"\n{arm_name.upper()}:")
    print(f"  Brier Score: {brier:.4f}  (lower = better calibration)")
    print(f"  Log Loss:    {logloss:.4f}  (lower = better)")
    print(f"  Accuracy:    {accuracy:.1%}  ({correct}/{len(df_arm)} correct at 0.5 threshold)")

print("\n" + "=" * 80)

# Summary table
df_summary = pd.DataFrame(results_summary)
print("\nSummary:")
df_summary

EVALUATION: Forecast Accuracy by Arm

BASELINE:
  Brier Score: 0.2868  (lower = better calibration)
  Log Loss:    0.7732  (lower = better)
  Accuracy:    50.0%  (7/14 correct at 0.5 threshold)

VOLUME:
  Brier Score: 0.3231  (lower = better calibration)
  Log Loss:    0.8530  (lower = better)
  Accuracy:    35.7%  (5/14 correct at 0.5 threshold)

MICROSTRUCTURE:
  Brier Score: 0.2887  (lower = better calibration)
  Log Loss:    0.7717  (lower = better)
  Accuracy:    33.3%  (4/12 correct at 0.5 threshold)


Summary:


,arm,brier_score,log_loss,accuracy,n_markets
0,baseline,0.286793,0.773186,0.500000,14
1,volume,0.323142,0.852957,0.357143,14
2,microstructure,0.288723,0.771675,0.333333,12


In [24]:
# Next steps checklist
print("""\n📋 NEXT STEPS:\n
1. [ ] Scale to all markets: Change N_MARKETS = 349
2. [ ] Replace fake outcomes with real API data
3. [ ] Run statistical significance tests between arms
4. [ ] Add visualization (calibration plots, etc.)
5. [ ] Experiment with different prompts/models
""")


📋 NEXT STEPS:

1. [ ] Scale to all markets: Change N_MARKETS = 349
2. [ ] Replace fake outcomes with real API data
3. [ ] Run statistical significance tests between arms
4. [ ] Add visualization (calibration plots, etc.)
5. [ ] Experiment with different prompts/models

